# SA3 LoRA Training — underfit on Colab

**v2 — every known trap already worked around.** Built 2026-09-13 after a full debugging
session; the fixes below are not optional, they are why this version works.

| | |
|---|---|
| Google account | `w2@2w12.one` — Colab Pro, Drive and HuggingFace must ALL be this one |
| Model | `sa3-medium` |
| Dataset | pre-encoded latents zip on Drive |
| Trigger word | `zvq` |

### Before you run anything

1. **Runtime → Change runtime type → L4 GPU → Save.** Do this FIRST — changing it later
   wipes the machine and you lose the 24 GB download.
2. Put your latents zip in **My Drive → Colab Notebooks**.
3. Click *Agree and access repository* once at
   <https://huggingface.co/stabilityai/stable-audio-3-medium> (gated; a token alone is not enough).

> ⚠️ **Finishing: Runtime → Disconnect and delete runtime.** Compute units burn while the
> VM is alive, training or not.

### Measured on an L4 (medium, 2048 crop)

| Batch | s/step | 10,000 samples |
|---|---|---|
| 1 | 2.29 | ~6.4 h |
| 4 | *measure it* | target ~3 h |

VRAM at batch 1 was 6.9 GB of 22.5 — the GPU is mostly idle, which is why batch 4 is worth it.


---
## 1 — GPU check

**Want:** `NVIDIA L4, 23034 MiB`. Anything else, fix the runtime type and re-run.


In [ ]:
import shutil, subprocess
if not shutil.which('nvidia-smi'):
    print('NO GPU — Runtime > Change runtime type > L4 GPU > Save, then re-run')
else:
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip())


---
## 2 — Install underfit + stable-audio-3

~5 GB of wheels. **Want:** `EXIT 0` and `underfit==0.1.0`.


In [ ]:
import os, subprocess
for url, dest in [('https://github.com/dada-bots/underfit', '/content/underfit'),
                  ('https://github.com/Stability-AI/stable-audio-3', '/content/stable-audio-3')]:
    if not os.path.isdir(dest):
        subprocess.run(['git','clone','--depth','1',url,dest], check=True)
r = subprocess.run(['./install.sh','--no-setup'], cwd='/content/underfit',
                   capture_output=True, text=True)
print(r.stdout[-600:]); print('EXIT', r.returncode)


---
## 3 — Mount Drive

`/content/drive/MyDrive`. **Everything not on Drive is deleted when the session ends.**
Authorise with **w2@2w12.one**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for f in sorted(os.listdir('/content/drive/MyDrive/Colab Notebooks'))[:20]:
    print(' ', f)


---
## 4 — HuggingFace token

Paste into the popup — it is never written into the notebook.

> If you get `GatedRepoError: 403` later, you skipped the *Agree and access repository*
> click. A valid token is **not** access.


In [ ]:
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('HuggingFace token: ')
print('token length', len(os.environ['HF_TOKEN']))


---
## 5 — Download the model (~24 GB)

base 14 GB (fine-tuned) + ARC 10 GB (renders demos). Several minutes.

> **Fix baked in:** the progress loop uses `ps | grep '[u]nderfit...'`. A plain
> `pgrep -f underfit.cli.setup` matches *its own* command line, so the loop never exits.


In [ ]:
import subprocess, os, time
subprocess.Popen('cd /content/underfit && nohup uv run python -m underfit.cli.setup '
                 '--backend sa3 --backend-path /content/stable-audio-3 --models sa3-medium '
                 '> /content/setup.log 2>&1 &', shell=True, env=dict(os.environ))

def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

while True:
    running = sh("ps -eo args | grep -c '[u]nderfit.cli.setup'") or '0'
    print(f"downloaded: {sh('du -sh /root/.cache/huggingface 2>/dev/null | cut -f1') or '0'}"
          f'   (running: {running})', flush=True)
    if running.strip() == '0': break
    time.sleep(30)

print(sh('tail -4 /content/setup.log'))
print('\nmodels:'); print(sh('ls -l /content/underfit/state/models/sa3-medium/'))


---
## 6 — Unpack the latents

**Want:** `38 latent files`. Edit `ZIP` if your dataset has a different name.

*Latents are your audio pre-compressed (~128x) into what the model trains on. mira's
captions live inside them, so this one file carries everything.*


In [ ]:
import subprocess, glob, os
ZIP = '/content/drive/MyDrive/Colab Notebooks/dune-ost-latents-same-l.zip'
LAT = '/content/latents'
assert os.path.exists(ZIP), f'Not found: {ZIP}'
os.makedirs(LAT, exist_ok=True)
subprocess.run(['unzip','-q','-o',ZIP,'-d',LAT], check=True)
sub = [d for d in glob.glob(LAT+'/*') if os.path.isdir(d)]
LATDIR = sub[0] if sub else LAT
print(len(glob.glob(LATDIR+'/*.npy')), 'latent files')
print('LATDIR =', LATDIR)


---
## 7 — Import the dataset in the dashboard

Run cell 8 first to start the dashboard, then come back here.

In the UI: **NEW DATASET** → paste the `LATDIR` path printed above.

| Field | Value |
|---|---|
| Dataset name | `dune-ost` (name it for the content, not the folder) |
| Encoder | **SA3-medium (latent_dim=256)** |

It will say *Pre-encoded latents detected* — that is correct, it will not re-encode.

**Then run cell 9 (the repair cell). The dataset will be broken until you do.**


---
## 8 — Start the dashboard

> **Two fixes baked in, both mandatory:**
> 1. `uv pip install safetensors>=0.8.0` — the SA3 install adds it, but `uv run` re-syncs
>    the venv to underfit's lockfile and **downgrades it back to 0.7.0**, after which
>    training dies with `ImportError: safetensors>=0.8.0 is required`.
> 2. The dashboard is launched with **`.venv/bin/python`, never `uv run`** — same reason.
>    underfit's own `run.sh` does exactly this.
>
> The venv has **no `pip`** (uv-built), so `uv pip install --python <venv>` is the only way.


In [ ]:
import subprocess, os, time
PY = '/content/underfit/.venv/bin/python'

subprocess.run("pkill -f 'dashboard/server.py'", shell=True); time.sleep(3)
subprocess.run(f"uv pip install --python {PY} 'safetensors>=0.8.0'", shell=True)
print('safetensors:', subprocess.run(
    f'{PY} -c "import safetensors; print(safetensors.__version__)"',
    shell=True, capture_output=True, text=True).stdout.strip())

env = dict(os.environ)
env.update({'UNDERFIT_STATE_DIR':'/content/underfit/state',
            'UNDERFIT_MODELS_DIR':'/content/underfit/state/models'})
subprocess.Popen('cd /content/underfit && nohup .venv/bin/python dashboard/server.py '
                 '> /content/dashboard.log 2>&1 &', shell=True, env=env)
time.sleep(20)
from google.colab import output
output.serve_kernel_port_as_window(8787)


---
## 9 — Repair the imported dataset ⚠️ REQUIRED

Run this **after** importing in the UI. Without it the run is broken in two ways:

**a) The dataset shows `error` forever.** Startup validation marks any `ready` dataset as
`error` when `latent_dir/details.json` is missing, and the import does not link it. Patching
`status` by hand looks like it works and then reverts on the next restart.

**b) Your captions are silently discarded.** The tag UI finds keys by walking an *audio*
`input_dir`; an imported latents dataset has none, so it shows *No files with tags found*,
posts `tag_keys: []`, and **every prompt collapses to just the trigger.** Training looks
completely normal. This is the dangerous one.

The cell gives underfit a directory of your real `.json` sidecars beside placeholder `.wav`
files (it requires >= 4096 bytes). Training never reads them — it reads latents.

> The dashboard must be **stopped** while `datasets.json` is edited: a live server holds the
> registry in memory and writes it back over your changes.


In [ ]:
import json, os, glob, shutil, struct, subprocess, time, wave

subprocess.run("pkill -f 'dashboard/server.py'", shell=True); time.sleep(4)
print('dashboards alive (want 0):', subprocess.run(
    "ps -eo args | grep -c '[d]ashboard/server.py'", shell=True,
    capture_output=True, text=True).stdout.strip())

TAG = '/content/tagdir'; os.makedirs(TAG, exist_ok=True)
sil = struct.pack('<h', 0) * 4096
for j in glob.glob(LATDIR + '/*.json'):
    stem = os.path.splitext(os.path.basename(j))[0]
    shutil.copy2(j, os.path.join(TAG, stem + '.json'))
    w = wave.open(os.path.join(TAG, stem + '.wav'), 'wb')
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(44100)
    w.writeframes(sil); w.close()
print('tagdir:', len(glob.glob(TAG+'/*.wav')), 'wav')

REG = '/content/underfit/state/datasets.json'
d = json.load(open(REG))
seq = d if isinstance(d, list) else list(d.get('datasets', d).values())
for ds in seq:
    if ds.get('status') and 'latent_dir' in ds:
        shadow = ds['latent_dir']
        src = os.path.join(LATDIR, 'details.json')
        if os.path.exists(src) and os.path.isdir(shadow):
            shutil.copy2(src, os.path.join(shadow, 'details.json'))
        ds['input_dir'] = TAG
        ds['status'] = 'ready'
        ds.pop('error', None)
        print('repaired:', ds['id'])
json.dump(d, open(REG, 'w'), indent=1)

for c in glob.glob('/content/underfit/state/**/*_tags.json', recursive=True):
    os.remove(c)

chk = json.load(open(REG))
s2 = chk if isinstance(chk, list) else list(chk.get('datasets', chk).values())
print('VERIFY:', [(x['input_dir'], x['status'], x['num_files']) for x in s2])
print('\nNow re-run cell 8 to restart the dashboard.')


---
## 10 — Auto-save to Drive ⭐ RUN THIS BEFORE LAUNCHING

**Colab can recycle your runtime at any time — Pro included.** Pro buys faster GPUs and
longer sessions, *not* a guaranteed machine. When it happens, `/content` is wiped: models,
latents, checkpoints, demos, all gone. It has already cost one run.

This starts a background watcher that copies every new checkpoint and demo to Drive within
60 seconds of it being written. After that a lost runtime costs ~10 minutes of re-setup
instead of hours of training.

Run it **before** you launch training. It keeps running while you use other cells.

| Goes to | |
|---|---|
| `Colab Notebooks/sa3-loras` | `.safetensors` checkpoints |
| `Colab Notebooks/sa3-demos` | demo audio |

> Files are only copied once their size has stopped changing, so a half-written checkpoint
> is never uploaded.


In [ ]:
import threading, glob, shutil, os

DEST_CK   = '/content/drive/MyDrive/Colab Notebooks/sa3-loras'
DEST_DEMO = '/content/drive/MyDrive/Colab Notebooks/sa3-demos'
os.makedirs(DEST_CK, exist_ok=True); os.makedirs(DEST_DEMO, exist_ok=True)

_seen, _sizes = set(), {}
_stop = threading.Event()

def _stable(f):
    """True once the file's size has stopped changing (not mid-write)."""
    try: s = os.path.getsize(f)
    except OSError: return False
    prev = _sizes.get(f)
    _sizes[f] = s
    return prev == s and s > 0

def _watch():
    while not _stop.is_set():
        try:
            for f in glob.glob('/content/underfit/state/runs/**/*.safetensors', recursive=True):
                if f not in _seen and _stable(f):
                    shutil.copy2(f, os.path.join(DEST_CK, os.path.basename(f)))
                    _seen.add(f); print('[autosave] ckpt', os.path.basename(f), flush=True)
            for pat in ('*.mp3', '*.wav'):
                for f in glob.glob(f'/content/underfit/state/runs/**/{pat}', recursive=True):
                    if f not in _seen and _stable(f):
                        rel = f.split('/runs/')[-1].replace('/', '__')
                        shutil.copy2(f, os.path.join(DEST_DEMO, rel))
                        _seen.add(f); print('[autosave] demo', rel, flush=True)
        except Exception as e:
            print('[autosave] warn:', e, flush=True)
        _stop.wait(60)

threading.Thread(target=_watch, daemon=True).start()
print('autosave ON  (checks every 60s)')
print('  checkpoints ->', DEST_CK)
print('  demos       ->', DEST_DEMO)
print('\nStop with:  _stop.set()')


---
## 11 — Settings

### New Finetune

| Field | Value |
|---|---|
| Run name | `dune-zvq-medium-01` |
| Base model | SA3-medium |
| Latent seq length | **2048** (190 s) |
| Crop mode | Random |
| LoRA type | DoRA-rows |
| Rank / Alpha | 16 / = rank |
| LR | 1e-4 |
| **Batch size** | **4** |
| **Max steps** | **2500** |
| Ckpt / Demo every | 500 / 500 |

> **Why batch 4 x 2500.** That is 10,000 samples — identical to batch 1 x 10,000, but the
> GPU is properly fed. Batch 1 used 6.9 GB of 22.5 and ran 2.29 s/step (~6.4 h).
> Leaving steps at 10,000 with batch 4 would be **4x the training**, not faster.
>
> **Why 2048 not medium's native 4096.** 4096 = 380 s; most cues are shorter, so ~29% of
> every step would be padding. 2048 fits 27 of 38 tracks fully.

### Dataset Text Prompts — the screen that silently ruins runs

| Setting | Value |
|---|---|
| Use fixed prompt | ❌ OFF |
| Prepend to prompt | ✅ `zvq`, 80% |
| Use tags | ✅ ON |
| shuffle | ✅ ON |
| Balance bar | **Tags 100%** |

**Leave ON — exactly 7:**
`TrackType` `VocalType` `genre` `instruments` `moods` `bpm` `keyscale`

**Turn OFF — 15:**
`audio_samples` `length_seconds` `path` `prompt` `relpath` `seconds_total` `src_relpath`
`trigger` `audio_dir` `codec` `count` `max_duration` `max_samples` `pad_modulo` `sample_rate`

> Pills come from the *latent* sidecars (pre-encode bookkeeping) plus `details.json`, not
> mira's clean caption. Left on, prompts train on absolute file paths and sample counts.
>
> **Preview check:** musical words only. No `/Users/...`, no `.npy`, no `6043649`.
>
> ⚠️ **Clone Settings does NOT preserve tag pills** — they reset to all-on. Redo this
> screen on every cloned run.

### Demos

Preset **Four**, all **ARC**, **CFG 1**, **Steps 8**, **2048 (190s)**, different seeds:

- `zvq, TrackType: Music, VocalType: Instrumental, Genre: Electronic: Ambient, Moods: dark, epic, film, Instruments: synthesizer, strings, low brass`
- `zvq, Moods: relaxing, film, Instruments: piano, strings, BPM: 64`
- `zvq, BPM: 102, Moods: action, epic`
- *(leave empty — unconditional control: if this starts sounding like your dataset, the LoRA is bleeding into the base model)*

Avoid Base demos (need ~50 steps each) and never Steps 2 — renders mush and looks like a broken LoRA.


---
## 12 — Status check

Run any time. tqdm redraws in place, so a live log **looks** frozen but isn't — trust this instead.


In [ ]:
import subprocess
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', sh('nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader'))
print('training:', 'YES' if sh("ps -eo args | grep -c '[l]ora_train'") not in ('0','') else 'no')
log = sh('ls -t /content/underfit/state/runs/*.log 2>/dev/null | head -1')
if log:
    print('\n--- last lines ---'); print(sh(f"tail -c 600 '{log}'"))
print('\n--- checkpoints ---')
print(sh('ls -1t /content/underfit/state/runs/*/*/checkpoints/*.safetensors 2>/dev/null | head -5') or '  none yet')


---
## 13 — Save LoRAs to Drive (manual backup) ⚠️

**Run before ending the session.** `/content` is wiped; Drive is not. Safe to run repeatedly.


In [ ]:
import glob, shutil, os
dest = '/content/drive/MyDrive/Colab Notebooks/sa3-loras'
os.makedirs(dest, exist_ok=True)
found = glob.glob('/content/underfit/state/runs/**/*.safetensors', recursive=True)
for f in found:
    shutil.copy2(f, os.path.join(dest, os.path.basename(f)))
    print('saved', os.path.basename(f))
print(f'\n{len(found)} checkpoint(s) -> {dest}')


---
## 14 — Emergency: kill a stuck run

The Colab port-proxy window goes stale and the dashboard stops responding — **KILL does
nothing**. Kill from here instead, then re-run cell 8 for a fresh dashboard link.
(Close the old browser tab; it will not recover.)


In [ ]:
import subprocess, time
subprocess.run("pkill -f 'lora_train'", shell=True); time.sleep(5)
print('trainers alive (want 0):', subprocess.run(
    "ps -eo args | grep -c '[l]ora_train'", shell=True,
    capture_output=True, text=True).stdout.strip())
print(subprocess.run('nvidia-smi --query-gpu=memory.used --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip())


---
## ⚠️ Finishing up

1. **Cell 13** — final manual save (autosave should already have them).
2. **Runtime → Disconnect and delete runtime.**

### Reading the run

| Signal | Healthy |
|---|---|
| Loss (smoothed) | drifts down then flattens — the flat part is the underfit sweet spot |
| Loss (raw) | very noisy. Normal: batch 1-4 + a random noise level each step |
| Grad norm | decays smoothly |
| LoRA magnitude | drifts off its start value. Dead flat = not learning. Spiking = LR too high |
| Unconditional demo | should NOT sound like your dataset |

The best checkpoint is often **not** the last one.

### Using the LoRA on your Mac

Medium inference needs ~5 GB and runs fine on 16 GB — train in the cloud, generate at home.
`lora_merge.py` reads underfit/PEFT torch safetensors directly, no conversion.

```bash
cd sa3-studio/stable-audio-3/optimized/mlx
.venv/bin/python scripts/sa3_mlx.py --dit medium \
  --lora ~/Downloads/dune-zvq.safetensors --lora-strength 0.7 \
  --prompt 'zvq, TrackType: Music, Moods: dark, epic, Instruments: strings, low brass' \
  --seconds 120 --out dune-test.wav
```
